In [ ]:
import os
import requests
from tqdm import tqdm

In [ ]:
BASE_URL = "https://dataverse.nl/api"
DOI = "doi:10.34894/AECRSD"

OUTPUT_ROOT = "WMH_data"

ALLOWED_PREFIXES = ("training/", "test/")

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Fetch dataset metadata
print("Fetching dataset metadata...")
r = requests.get(
    f"{BASE_URL}/datasets/:persistentId/versions/:latest",
    params={"persistentId": DOI},
)
r.raise_for_status()

files = r.json()["data"]["files"]
print(f"Found {len(files)} total files in dataset")

# Filter training + test
selected = []
for f in files:
    directory = f.get("directoryLabel", "")
    if directory.startswith(ALLOWED_PREFIXES):
        selected.append(f)

print(f"Will download {len(selected)} files into {OUTPUT_ROOT}/")


In [ ]:
for f in tqdm(selected, desc="Downloading WMH files"):
    directory = f["directoryLabel"]              # e.g. training/Amsterdam/GE3T/100
    filename = f["dataFile"]["filename"]         # e.g. FLAIR.nii.gz
    file_id = f["dataFile"]["id"]

    # Preserve directory structure under WMH_data/
    local_dir = os.path.join(OUTPUT_ROOT, directory)
    os.makedirs(local_dir, exist_ok=True)

    local_path = os.path.join(local_dir, filename)

    # Skip already-downloaded files
    if os.path.exists(local_path):
        continue

    url = f"{BASE_URL}/access/datafile/{file_id}"
    with requests.get(url, stream=True) as rf:
        rf.raise_for_status()
        with open(local_path, "wb") as out:
            for chunk in rf.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    out.write(chunk)


In [ ]:
for split in ["training", "test"]:
    path = os.path.join("WMH_data", split)
    print(split, "exists:", os.path.exists(path))